# First ML model: classification with scikit-learn

In [1]:
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

df = sns.load_dataset('titanic')
num = ['age', 'fare', 'pclass']
cat = ['sex', 'embarked']
X, y = df[num + cat], df['survived']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [2]:
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat),
])
model = Pipeline([('pre', pre), ('clf', RandomForestClassifier(random_state=42))])
print(cross_val_score(model, X_tr, y_tr, cv=5).mean())
model.fit(X_tr, y_tr)
print(classification_report(y_te, model.predict(X_te)))

0.8006599034768049
              precision    recall  f1-score   support

           0       0.83      0.86      0.85       110
           1       0.77      0.72      0.75        69

    accuracy                           0.81       179
   macro avg       0.80      0.79      0.80       179
weighted avg       0.81      0.81      0.81       179



## Example 3: Compare with logistic regression
A simpler linear model gives us a baseline to compare with the random forest.

In [3]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ('pre', pre),
    ('clf', LogisticRegression(max_iter=1000, random_state=42)),
])

logistic_score = cross_val_score(logistic_model, X_tr, y_tr, cv=5).mean()
print(f'Logistic regression CV accuracy: {logistic_score:.3f}')

Logistic regression CV accuracy: 0.796


## Example 4: Tune a random forest with GridSearchCV

**Model settings**

A random forest combines many decision trees and uses their majority prediction (classification) or average prediction (regression). It is controlled by **hyperparameters**, which are settings chosen before training. Two important random forest hyperparameters are:

- `n_estimators`: the number of trees in the forest.
- `max_depth`: the maximum depth of each tree. A deeper tree can learn more detail, but it may overfit by memorizing the training data.

**Hyperparameter tuning**

Hyperparameter tuning searches for settings that perform well on unseen data. `GridSearchCV` tries every combination supplied in `param_grid`. Here it will test:

- 2 values for `n_estimators`: 50 or 100
- 3 values for `max_depth`: `None`, 5, or 10
- 6 combinations in total

The `clf__` prefix means that the setting belongs to the pipeline step named `clf`, not to the preprocessing step.

**Cross-validation**

For every combination, 5-fold cross-validation divides the training data into 5 parts. The model trains on 4 parts and validates on the remaining part. This repeats 5 times, so 30 model fits are performed (6 combinations x 5 folds). The combination with the highest mean validation accuracy becomes the best model.

`search.fit(...)` automatically retrains the best combination on the complete training set. `search.best_estimator_` is therefore ready to evaluate on the test set.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'clf__n_estimators': [50, 100],
    'clf__max_depth': [None, 5, 10],
}

search = GridSearchCV(model, param_grid, cv=5)
search.fit(X_tr, y_tr)
print(search.best_params_)
print(f'Best CV accuracy: {search.best_score_:.3f}')

# Finally measure the selected model on data it did not use during tuning.
test_score = search.best_estimator_.score(X_te, y_te)
print(f'Test accuracy: {test_score:.3f}')

## Exercises
1. Which model is easier to interpret: logistic regression or random forest?
2. Add features (family size, title) and measure the change.
3. Print the test-set classification report for `search.best_estimator_`.